# Nhập run train trên máy GPU riêng vào `runs_aug1`

1. Tải các file `aug1_*.zip` và `refit1_*.zip` từ `pc_runs/upload/` của máy riêng lên `MyDrive/Sym/upload/`.
2. Chọn runtime **CPU** là đủ. Trong 🔑 Secrets bật **Notebook access** cho `GH_TOKEN`.
3. Chạy **Setup** (đã đặt `IMAGES = False`, không giải nén ảnh), rồi **cell I**.

Cell I không ghi đè run đã có trên Drive và không đụng test. Zip bị từ chối nằm ở `upload/rejected/`: gửi Claude dòng thông báo của zip đó.

**23/11: cả 18 run (a) và refit train trên máy 5090.** Trước lần nhập đầu tiên: dừng cell G/K trên Colab, đổi tên `MyDrive/Sym/runs_aug1` thành `runs_aug1_colab` (và `runs_refit1` thành `runs_refit1_colab` nếu có). Cell I không nhập zip train trên GPU khác GPU của run đã có trên Drive.

In [ ]:
import os, sys, json, time, base64, shutil, subprocess
from pathlib import Path
from google.colab import drive, userdata

if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/Letri6102/Thesis_Sym.git'
BRANCH   = 'main'                      # nhánh stage1-ema-valchecks đã merge vào main (21/11)
REPO     = Path('/content/Thesis_Sym')
CODE     = REPO / 'Contrastive_V1'
SYM      = Path('/content/drive/MyDrive/Sym')
SPLITS   = SYM / 'splits_v2'           # manifest mới; splits/ cũ chỉ cho runs_retry1
RUNS     = SYM / 'runs_aug1'           # run mới với strong_train_aug
LOGS     = SYM / 'logs'
IMAGES   = False                       # False: không giải nén ảnh (phiên CPU chỉ chạy Tầng 2, cell E)

tok  = userdata.get('GH_TOKEN')
auth = 'http.extraHeader=Authorization: Basic ' + base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
def git(*args, cwd=None):
    r = subprocess.run(['git', '-c', auth, *args], cwd=cwd, capture_output=True, text=True,
                       env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    if r.returncode:
        raise RuntimeError(f'git {args[0]} lỗi (exit {r.returncode}): {r.stderr.strip()}')
try:
    if (REPO / '.git').exists():
        # refspec đầy đủ: bản clone 1 nhánh vẫn đổi được sang nhánh khác (fetch tên trơn chỉ cập nhật FETCH_HEAD)
        git('fetch', '-q', '--depth', '1', 'origin', f'+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}', cwd=REPO)
        git('reset', '-q', '--hard', f'origin/{BRANCH}', cwd=REPO)
    else:
        git('clone', '-q', '--depth', '1', '-b', BRANCH, REPO_URL, str(REPO))
finally:
    del tok, auth
print('commit:', subprocess.run(['git', 'log', '-1', '--format=%h %s'], cwd=REPO,
                                capture_output=True, text=True).stdout.strip())

SHARDS = SYM / 'DISFA_img'                     # 27 tar theo subject, do cell A tạo
if IMAGES and (SHARDS / 'DONE.json').exists():
    DATA = Path('/content/DISFA')
    if not (DATA / '.img_ok').exists():        # chưa giải nén, hoặc lần trước giải nén dở
        t0, meta = time.time(), json.loads((SHARDS / 'DONE.json').read_text())
        DATA.mkdir(parents=True, exist_ok=True)
        print(f'Giải nén {len(meta["subjects"])} tar ({meta["bytes"] / 1e9:.1f} GB) về /content:', flush=True)
        for i, s in enumerate(meta['subjects'], 1):
            subprocess.run(['tar', '-xf', str(SHARDS / f'{s}.tar'), '-C', str(DATA)], check=True, timeout=1800)
            print(f'  {i:>2}/{len(meta["subjects"])} {s}  {(time.time() - t0) / 60:.1f} phút', flush=True)
        n = sum(len(f) for _, _, f in os.walk(DATA / 'img'))
        assert n == meta['n_images'], f'Giải nén được {n} ảnh, cần {meta["n_images"]}'
        shutil.copytree(SYM / 'DISFA' / 'list', DATA / 'list', dirs_exist_ok=True)
        (DATA / '.img_ok').touch()
        print(f'Đã giải nén {n} ảnh về /content trong {time.time() - t0:.0f}s')
else:
    DATA = SYM / 'DISFA'                 # đọc thẳng Drive: chậm và hay lỗi FileNotFoundError
print('DATA =', DATA)

os.chdir(CODE)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
(CODE / 'checkpoints').mkdir(exist_ok=True)
w_src = SYM / 'Contrastive_V1' / 'checkpoints' / 'resnet50-19c8e357.pth'
w_dst = CODE / 'checkpoints' / 'resnet50-19c8e357.pth'
if w_src.exists() and not w_dst.exists():
    shutil.copy(w_src, w_dst)
import torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| CPU', os.cpu_count(),
      '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ')

import re
def run_logged(cmd, name):
    """Chạy lệnh con, ghi toàn bộ output vào MyDrive/Sym/logs/ và in các dòng đáng xem.
    (Colab không hiện stdout của tiến trình con nếu chỉ dùng subprocess.run.)"""
    LOGS.mkdir(exist_ok=True)
    log = LOGS / f'{time.strftime("%Y%m%d_%H%M%S")}_{name}.log'
    print(f'>> {name} | log: {log.name}', flush=True)
    with open(log, 'w') as f, subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                                               stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout:
            f.write(line); f.flush()
            if line.startswith('{"epoch"'):
                r = json.loads(line)
                print(f"   ep{r['epoch']:>2}  wa={r['train']['wa']:.4f}  valF1={r['val']['macro_f1']:.3f}"
                      f"  {r['seconds'] / 60:.0f} phút", flush=True)
            elif line.startswith('epoch ') and ' step ' in line:
                if re.search(r'step (1000|2000)/', line):
                    print('   ' + line.rstrip(), flush=True)
            elif 'FutureWarning' not in line and 'warnings.warn' not in line:
                print('   ' + line.rstrip(), flush=True)
    if p.returncode:
        raise RuntimeError(f'{name} dừng với mã {p.returncode}, xem log {log}')

In [ ]:
# ==== Cell I: nhập run train trên máy GPU riêng (MyDrive/Sym/upload/*.zip) — kiểm khoá, không ghi đè ====
import re, zipfile
sys.path.insert(0, str(CODE))
from stage2_decoder.contrast_ci import LOCK, config_problems, run_problems
from refit import run as R

UP, BUSY_MIN = SYM / 'upload', 15
REFIT = SYM / 'runs_refit1'
assert RUNS.name == 'runs_aug1', f'RUNS phải là runs_aug1, đang là {RUNS}'
REFIT_HASH = R.write_manifests(SPLITS, DATA, REFIT)      # manifest refit chuẩn, tính lại từ splits_v2 đã khoá

def ok_aug1(run, v, k, s):
    return not run_problems(run, v, k, s, LOCK) and (run / 'last.pt').exists()

def ok_refit1(run, v, k, s):
    try:
        rec, cfg = (json.loads((run / n).read_text()) for n in ('refit.json', 'config.json'))
    except (OSError, ValueError):
        return False
    a = RUNS / v / f'fold{k}' / f'seed{s}'
    lock = dict(LOCK, manifests={**LOCK['manifests'], k: REFIT_HASH[k]})
    return (rec['manifest_hash'] == REFIT_HASH[k] and R.n_epochs(run) == rec['E'] and (run / 'last.pt').exists()
            and not config_problems(cfg, v, k, s, lock) and ok_aug1(a, v, k, s) and R.selected_epoch(a) == rec['E'])

KINDS = {'aug1': (RUNS, ok_aug1), 'refit1': (REFIT, ok_refit1)}

def park(z, sub):
    (UP / sub).mkdir(parents=True, exist_ok=True)
    shutil.move(str(z), str(UP / sub / z.name))

zips = sorted(UP.glob('*.zip'))                           # aug1_* đứng trước refit1_*: refit cần run (a) đã có
print(f'{len(zips)} zip chờ nhập trong {UP}', flush=True)
for z in zips:
    m = re.fullmatch(r'(aug1|refit1)_(baseline|full)_fold([123])_seed([012])', z.stem)
    if not m:
        print(f'  {z.name}: tên lạ, bỏ qua'); continue
    kind, v, k, s = m[1], m[2], int(m[3]), int(m[4])
    base, ok = KINDS[kind]
    target, prefix = base / v / f'fold{k}' / f'seed{s}', f'{v}/fold{k}/seed{s}/'
    work = Path('/content/pc_import') / z.stem
    shutil.rmtree(work, ignore_errors=True)
    work.mkdir(parents=True)
    subprocess.run(['cp', str(z), str(work / 'run.zip')], check=True, timeout=1800)     # đọc Drive có timeout
    with zipfile.ZipFile(work / 'run.zip') as f:
        if any(not n.startswith(prefix) or '..' in n.split('/') for n in f.namelist()):
            print(f'  {z.name}: zip chứa đường dẫn lạ → KHÔNG nhập'); park(z, 'rejected'); continue
        f.extractall(work / 'x')
    run = work / 'x' / v / f'fold{k}' / f'seed{s}'
    if not ok(run, v, k, s):
        print(f'  {z.name}: lệch khoá, chưa xong, hoặc thiếu run (a) tương ứng → KHÔNG nhập'); park(z, 'rejected'); continue
    gpu = json.loads((run / 'environment.json').read_text()).get('gpu')
    mixed = sorted({json.loads(p.read_text()).get('gpu') or '?' for b in (RUNS, REFIT)
                    for p in b.glob('*/fold*/seed*/environment.json') if p.parent != target} - {gpu})
    if mixed:                                             # 23/11: cả 18 run (a) + refit phải train trên cùng một máy
        print(f'  {z.name}: train trên {gpu}, nhưng Drive đã có run train trên {mixed} → để lại. '
              f'Dừng cell G/K, đổi tên {RUNS.name} và {REFIT.name} cũ thành *_colab, rồi chạy lại cell'); continue
    if (target / 'config.json').exists():
        if ok(target, v, k, s):
            print(f'  {z.name}: Drive đã có run này hoàn chỉnh → giữ bản Drive, không ghi đè'); park(z, 'duplicate'); continue
        files = [p for p in target.rglob('*') if p.is_file()]
        age = (time.time() - max(p.stat().st_mtime for p in files)) / 60 if files else 1e9
        if age < BUSY_MIN:
            print(f'  {z.name}: run này vừa được ghi {age:.0f} phút trước (Colab có thể đang train) → để lại, chạy lại cell sau'); continue
        dest = SYM / 'failed_runs' / f'{kind}_{v}_fold{k}_seed{s}_{time.strftime("%Y%m%d_%H%M%S")}_truoc_nhap'
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(target), str(dest))
        print(f'  bản dở trên Drive → failed_runs/{dest.name}')
    partial = target.with_name(target.name + '.partial')
    shutil.rmtree(partial, ignore_errors=True)
    partial.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['cp', '-r', str(run), str(partial)], check=True, timeout=3600)
    partial.rename(target)
    assert ok(target, v, k, s), f'{z.name}: sau khi chép lên Drive lại lệch, gửi Claude xem'
    park(z, 'done')
    shutil.rmtree(work)
    print(f'  {z.name}: ĐÃ NHẬP (GPU {gpu})', flush=True)

for kind, (base, ok) in KINDS.items():
    n = sum(ok(base / v / f'fold{k}' / f'seed{s}', v, k, s) for v in ('baseline', 'full') for k in (1, 2, 3) for s in (0, 1, 2))
    print(f'{base.name}: {n}/18 run hoàn chỉnh đúng khoá')
print('Đang chờ Drive đồng bộ...', flush=True)
drive.flush_and_unmount()

## Cell J — chạy MỘT lần, trước khi train trên máy GPU riêng

Không cần chạy Setup. Cell đối chiếu 27 tar của gói `LVTS_TriLe` với `DISFA_img` trên Drive; phải báo `KHỚP 27/27`.

In [ ]:
# ==== Cell J: đối chiếu 27 tar của gói LVTS_TriLe (dựng từ img/ ở máy cá nhân) với DISFA_img trên Drive — CPU, không cần Setup ====
import os, json, hashlib, subprocess
from pathlib import Path
from google.colab import drive

if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
SRC, TMP = Path('/content/drive/MyDrive/Sym/DISFA_img'), Path('/content/tar_check')
TMP.mkdir(exist_ok=True)
EXPECT = {                                  # pc/bundle_tar_sha256.json, gói dựng từ code 4e610e2
    'SN001': (338176000, '393a2000c5f22dcb42d78d9b25a2e585f03ed1a13c676a287426331f0f353c93'),
    'SN002': (369029120, 'e8a01a3d749473be3ff95b55b05a88ec697bb4be651ef6ebd96ee2bdbf40fb16'),
    'SN003': (340551680, 'b8c52330619371cdf378090667309791b792aa7260580c77b3d2853c27a82f3a'),
    'SN004': (347494400, 'cda7a8a30e0860a288f6512ea2f116c14f71337479e8bc14369b53f562bf3191'),
    'SN005': (310538240, 'e523ed014ea8d5571f5598b1848f2f54f3c13823cd3e96f9ae1a04c97daaddbd'),
    'SN006': (313845760, 'e5b98bc6dc59eb80f259cc29bf4ac4a1189f9de52a92a168cba03eb701905ff2'),
    'SN007': (302182400, '45b100926ab2bf42850cc8846b2d32e3fafb9446bb543caad6ef6cb2bd1de7c4'),
    'SN008': (316416000, '9ce9036866b3a1699d912b99fb2527db616e52dfc4b280468623a89852cb4168'),
    'SN009': (323205120, 'aaca676e856301459ddc438c67ba215e621df72d16044d04e5a13ea6a2863d72'),
    'SN010': (320337920, '7d6489c3e27abf2af1892062a708d46571a16290bd740e5097be4d3a641d32a3'),
    'SN011': (330147840, '7fd49d6fed1cbba84d366d78dd16bbabe2da29085963f33decdfb50c521fd019'),
    'SN012': (370083840, '4995de042323479ea59863c53536c18c6d60e8706b897ce02ace7aefc8ab4072'),
    'SN013': (306667520, '3f879b0edd53c6a1597595d46f96c9a90c5206cf67908dca06eb43411c01bd0e'),
    'SN016': (379463680, 'd5137b37fbf4697f3311282e133571edc227bd8eaddb1898034b3d405df8e0c2'),
    'SN017': (324300800, '46a946bbe3a79ac35cee0993a96e2815f54050d10f413c69151d1b8edc35dbfa'),
    'SN018': (358594560, '21cbc64024545cfa25481d40adb4081481de230926e23d3df27d31dd0b998a0a'),
    'SN021': (296212480, 'b6aefa68d5aebf2358dad6838221b4bb7f12e27f9b097813ec1a44aa0b518484'),
    'SN023': (292945920, '13403558a95a7233506f80acd9f758010493a2b0b727fc629d57d640a29e2027'),
    'SN024': (315494400, '773b8c9390d7c58b00f94c650e895061f7e5ced2649d2f5c797c0172e99dc3fb'),
    'SN025': (307333120, 'b62af11e89c14e4cea1a8a7a4d3b32987fa347f9b5428e3719e39d9820f91e7f'),
    'SN026': (301824000, 'b8ad68fa146c0292f80ccfb15952c0bb41b494a3043329be3ed59f19b2d3a18a'),
    'SN027': (290570240, '270d34064cdae28a227755f81f5fbb72f95049218e680429d5c119d13e65c70e'),
    'SN028': (309800960, 'b049cae37e948e47d8aa47dadb9f664c6a08ccd72cc8ae7d1877967c9a372beb'),
    'SN029': (304179200, '5db9d64ff0eb49ae84e2adb9cba3a5764a76fce053c9f86389e894e27c52cb18'),
    'SN030': (266752000, '31617046c885f52da5a4d73f88665c5ecf106417002dc8abb8408ce24fce92ec'),
    'SN031': (313640960, '796902bef88cee76bcfd20e428d740882dea267ca6823794cf1a7375ed4a07dc'),
    'SN032': (302254080, '7743fe12496dcf9a7a18f1a5005fb954ce7855fd4811789262fac41ddaaee215'),
}

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

assert sorted(json.loads((SRC / 'DONE.json').read_text())['subjects']) == sorted(EXPECT), 'Danh sách subject khác'
bad = []
for i, (s, (size, want)) in enumerate(sorted(EXPECT.items()), 1):
    if json.loads((SRC / f'{s}.json').read_text())['size'] != size:
        bad.append(s); print(f'{i:>2}/27 {s}: KHÁC (kích thước)', flush=True); continue
    local = TMP / f'{s}.tar'
    subprocess.run(['cp', str(SRC / f'{s}.tar'), str(local)], check=True, timeout=1800)   # đọc Drive có timeout
    got = sha256(local)
    local.unlink()
    print(f'{i:>2}/27 {s}: ' + ('KHỚP' if got == want else 'KHÁC'), flush=True)
    if got != want:
        bad.append(s)
print('\nKẾT QUẢ: ' + ('KHỚP 27/27: ảnh trong gói đúng bản trên Drive, máy GPU được train.' if not bad
                       else f'KHÁC ở {bad}: KHÔNG train bằng gói này, báo Claude.'))